# Conformers

In [ ]:
import sys, os
sys.path.insert(0, 'optimal-conformer-encodings')

os.environ["TORCH_COMPILE_DISABLE"] = "1"

import importlib

import numpy as np
from McUtils.ExternalPrograms import SMILESSupplier, RDMolecule, SMILESTokenizer, ConformerEncoder
from McUtils.Data import UnitsData
import McUtils.Devutils as dev
from Psience.Molecools import Molecule
import McUtils.Iterators as itut
import McUtils.Scaffolding as scaff
import McUtils.Plots as plt
import McUtils.Coordinerds as coordops
import McUtils.Numputils as nput
import McUtils.Jupyter as interactive

import itertools

import ocen.validators as validators

In [ ]:
mol = Molecule.from_string(supp.find_smi(100), 'smi', energy_evaluator='aimnet2')

In [ ]:
mol.animate_mode(
    -1,
    backend='mesh3D'
).savefig('/Users/Mark/Desktop/anim_ppt.glb', 
    skinned=True)

#### Data Loading

In [ ]:
supp = SMILESSupplier.from_name('emols')

In [ ]:
max_smi_num = len(supp)

In [ ]:
np.random.seed(12321)
samps = np.random.choice(max_smi_num, size=100)

In [ ]:
smis = [
    supp.find_smi(s)
    for s in samps
]

In [ ]:
csmis = [Molecule.from_string(s, 'smi',).to_string('smi', remove_hydrogens=True) for s in smis]

In [ ]:
def load_data(mol_data, i, *keys, byte_size=16, encoding='compressed'):
    t = mol_data[i][byte_size][encoding]
    for k in keys:
        t = t[k]
    return t

#### Plotting

In [ ]:
PALETTE = (
    "#FFB257",
    "#9BCB52",
    "#85BADD",
    "#EF7F70",
    "#B985BD",
    "#c9ac28"
)
PALETTE = PALETTE + PALETTE + PALETTE

In [ ]:
def plot_highlight_mol(mol, overlap_lists, nx, **etc):
    sty = {}
    colors = [
        plt.prep_color('green', lighten=-2), 
        plt.prep_color('orange', lighten=-.5), 
        'purple'
    ]
    figure = None
    sty = {}
    b_sty = {}
    for i,ol in enumerate([ol for ol in overlap_lists if len(ol) > 1]):
        inds = [oo[nx] for oo in ol]
        for j in inds:
            sty[j] = {'glow': colors[i]}
        for j1,j2 in itertools.combinations(inds, 2):
            b_sty[(j1, j2)] = {'glow': colors[i]}
    return mol.plot(
        **(
            dict(
                backend='x3d',
                atom_style=sty,
                bond_style=b_sty
            ) | etc
        )
    )
def plot_highlight_mol2(mol, overlap_lists, nx, **etc):
    sty = {}
    colors = [
        plt.prep_color('green', lighten=-2), 
        plt.prep_color('orange', lighten=-.5), 
        'purple'
    ]
    figure = None
    sty = {}
    b_sty = {}
    figure = mol.plot(
        **(
            dict(
                backend='x3d'
            ) | etc
        )
    )
    highlight_counts = {}
    for i,ol in enumerate([ol for ol in overlap_lists if len(ol) > 1]):
        inds = [oo[nx] for oo in ol]
        for j in inds:
            highlight_counts[j] = highlight_counts.get(j, 0) + 1
    for i,ol in enumerate([ol for ol in overlap_lists if len(ol) > 1]):
        inds = [oo[nx] for oo in ol]
        transparencies = {
            int(k):{'color':colors[i], 'transparency':(1 - .6/highlight_counts[j])}
            for k,j in enumerate(inds)
        }
        mol.take_submolecule(inds).plot(
            backend='x3d',
            atom_radii=.7,
            atom_radius_scaling=1,
            bond_radius=.5,
            render_multiple_bonds=False,
            render_fractional_bonds=False,
            atom_style=transparencies,
            bond_style=False,#{'color':colors[i], 'transparency':.2},
            split_bond_segments=False,
            figure=figure
        )
    return figure
def plot_chain_decomp(mol, atom_labels=None, prep_atom_labels=False, **etc):
    m = mol.remove_hydrogens()
    frags = m.edge_graph.get_canonical_fragments(np.arange(len(m.atoms)))
    overlap_lists = []
    for f in frags:
        overlap_lists.append(f[1])
    colors = PALETTE
    sty = {}
    for i,ol in enumerate(overlap_lists):
        for oo in ol:
            sty[int(oo)] = {'glow': colors[i]}
    if atom_labels is None and prep_atom_labels:
        fragment_colors = {i:s['glow'] for i,s in sty.items()}
        atom_labels, xy = make_atom_labels(m, fragment_colors)
    if prep_atom_labels:
        return atom_labels
    return m.plot(
        **(
            dict(
                backend='rdkit',
                bond_radius=.4,
                atom_radii=.33,
                atom_style=sty,
                atom_highlights_are_circles=True,
                # draw_implicit_hydrogens=False,
                blend_mixed_bonds=False,
                atom_labels=atom_labels
            ) | etc
        )
    )
def plot_match_comp(m1, m2, all_matches, **opts):
    return interactive.Grid([
        [
            plot_highlight_mol(
                m1,
                # all_matches[1].matches,
                all_matches,
                0, **opts
            ).to_widget(),
            plot_highlight_mol(
                m2,
                # all_matches[1].matches,
                all_matches,
                1, **opts
            ).to_widget()
        ]
    ])
def plot_match_comp2(m1, m2, all_matches, **opts):
    return interactive.Grid([
        [
            plot_highlight_mol2(
                m1,
                # all_matches[1].matches,
                all_matches,
                0, **opts
            ).to_widget(),
            plot_highlight_mol2(
                m2,
                # all_matches[1].matches,
                all_matches,
                1, **opts
            ).to_widget()
        ]
    ])
def plot_decomp_comp2(m1, m2, **opts):
    return interactive.Grid([
        [
            plot_chain_decomp(
                m1, **opts
            ).to_widget(),
            plot_chain_decomp(
                m2, **opts
            ).to_widget()
        ]
    ])

### Drug Examples

In [ ]:
import plot_lipitor_partition as plp
import smiles_partition_trace as spt
import smiles_tag_prefix_trace as tt
import pandas as pd


plp.PALETTE = PALETTE

In [ ]:
samp_drugs = drugs = [
    {
        "name": "Penicillin G",
        "smiles": "CC1([C@@H](N2[C@H](S1)[C@@H](C2=O)NC(=O)CC3=CC=CC=C3)C(=O)O)C",
    },
    {
        "name": "Streptomycin",
        "smiles": "C[C@H]1[C@@]([C@H]([C@@H](O1)O[C@@H]2[C@H]([C@@H]([C@H]([C@@H]([C@H]2O)O)N=C(N)N)O)N=C(N)N)O[C@H]3[C@H]([C@@H]([C@H]([C@@H](O3)CO)O)O)NC)(C=O)O",
    },
    {
        "name": "Tetracycline",
        "smiles": "C[C@@]1([C@H]2C[C@H]3[C@@H](C(=O)C(=C([C@]3(C(=O)C2=C(C4=C1C=CC=C4O)O)O)O)C(=O)N)N(C)C)O",
    },
    {"name": "Acetaminophen", "smiles": "CC(=O)NC1=CC=C(C=C1)O"},
    {
        "name": "Chlorpromazine",
        "smiles": "CN(C)CCCN1C2=CC=CC=C2SC3=C1C=C(C=C3)Cl",
    },
    {
        "name": "Norethindrone",
        "smiles": "C[C@]12CC[C@H]3[C@H]([C@@H]1CC[C@]2(C#C)O)CCC4=CC(=O)CC[C@H]34",
    },
    {
        "name": "Methotrexate",
        "smiles": "CN(CC1=CN=C2C(=N1)C(=NC(=N2)N)N)C3=CC=C(C=C3)C(=O)N[C@@H](CCC(=O)O)C(=O)O",
    },
    {
        "name": "Prednisone",
        "smiles": "C[C@]12CC(=O)[C@H]3[C@H]([C@@H]1CC[C@@]2(C(=O)CO)O)CCC4=CC(=O)C=C[C@]34C",
    },
    {
        "name": "Warfarin",
        "smiles": "CC(=O)CC(C1=CC=CC=C1)C2=C(C3=CC=CC=C3OC2=O)O",
    },
    {
        "name": "Propranolol",
        "smiles": "CC(C)NCC(COC1=CC=CC2=CC=CC=C21)O",
    },
    {"name": "Metformin", "smiles": "CN(C)C(=N)N=C(N)N"},
    {
        "name": "Fluoxetine",
        "smiles": "CNCCC(C1=CC=CC=C1)OC2=CC=C(C=C2)C(F)(F)F",
    },
    {
        "name": "Zidovudine",
        "smiles": "CC1=CN(C(=O)NC1=O)[C@H]2C[C@@H]([C@H](O2)CO)N=[N+]=[N-]",
    },
    {
        "name": "Omeprazole",
        "smiles": "CC1=CN=C(C(=C1OC)C)CS(=O)C2=NC3=C(N2)C=C(C=C3)OC",
    },
    {
        "name": "Simvastatin",
        "smiles": "CCC(C)(C)C(=O)O[C@H]1C[C@H](C=C2[C@H]1[C@H]([C@H](C=C2)C)CC[C@@H]3C[C@H](CC(=O)O3)O)C",
    },
]

In [ ]:
import plot_lipitor_partition as plp

In [ ]:
plp.PALETTE = PALETTE
partition = plp.prepare_partition()
fig = plp.draw_partition(
    partition,
    default_atom_label_offset=.3,
    atom_label_adjustments={
        0: {"offset": (-1.05, .5)},
        1: {"offset": (-.1, 0)},
        17: {"offset": (-.7, -.9)},
        18: {"offset": (1, -.2)},
        
        20: {"offset": (.1, .3)},
        31: {"offset": (-.1, -.3)},
        32: {"offset": (.2, .1)},
        34: {"offset": (0, .4)},
    },
    draw_coords_options={'a1': {'scaling': None, 'radius':.9}},
    draw_label_adjustments={
        "r1": {"offset": (.4, -2)},
        "r2": {"offset": (0.0, .6)},
        "a1": {"offset": (-1.2, 2.8)},
    },
)

fig#.to_widget().write('lipitor.svg')

In [ ]:
def test_decomp(smi, display_atom_numbers=True):
    return plot_chain_decomp(
        Molecule.from_string(
            Molecule.from_string(
                smi,
                'smi'
            ).to_string('smi', include_tag=True, canonical=True, remove_hydrogens=True),
            'smi'
        ),
        display_atom_numbers=display_atom_numbers,
        image_size=800
    )

In [ ]:
drugs_by_name = {
    s['name']:s for s in drugs
}

In [ ]:
# for s in drugs:
#     try:
#         test_decomp(s['smiles'], display_atom_numbers=False).to_widget().write(
#             'mols_no_num/'+s['name']+".svg"
#         )
#     except:
#         print(s['name'])

#### Actual Renders

In [ ]:
drugs2 = dev.read_json('drugs.json')

In [ ]:
drug_smis2 = pd.read_csv('FDA_Approved_structures.csv')

In [ ]:
_, _, _, x, y = nput.intersection(
    drugs2,
    drug_smis2['Name'],
    return_indices=True
)
drugs_by_name2 = dict(zip(
    drug_smis2['Name'][y],
    drug_smis2['SMILES'][y]
))
    

In [ ]:
# for k,s in drugs_by_name2.items():
#     try:
#         test_decomp(s, display_atom_numbers=False).to_widget().write('mols_no_num/'+k+".svg")
#     except:
#         print(k)

In [ ]:
def make_smi(smi, include_tag=True, sel=None):
    mol = Molecule.from_string(smi, 'smi',
                               confgen_opts={
                                   'random_seed':254
                               })
    smi = mol.to_string(
        'smi',
        include_tag=include_tag,
        canonical=True,
        remove_hydrogens=True
    )
    if sel is not None:
        mol = mol.from_string(smi, 'smi')
        mol = mol.take_submolecule(sel)
        smi = mol.to_string(
            'smi',
            include_tag=include_tag,
            canonical=True,
            remove_hydrogens=True
        )
    return smi

In [ ]:
plp.PALETTE = PALETTE
plp.draw_partition(
    drugs_by_name2['Fluoxetine'], 
    display_atom_numbers=None,
    atom_labels=None,
    single_atom_palette=PALETTE[-3:]
)

In [ ]:
plp.PALETTE = PALETTE
plp.draw_partition(
    drugs_by_name2['Warfarin'], 
    display_atom_numbers=None,
    atom_labels=None,
    single_atom_palette=PALETTE[-3:]
)#.to_widget().write('mols_no_num/'+'Warfarin'+".svg")

In [ ]:
steps = tt.trace_tag_prefixes(spt.make_smi(drugs_by_name2['Fluoxetine']))
tt.export_tag_prefixes_pptx(
    steps,
    "mols_text/Fluoxetine_tag.pptx",
    palette=plp.PALETTE,
    single_atom_palette=PALETTE[-3:]
    # single_atom_color=plp.SINGLE_ATOM_COLOR,
)

In [ ]:
steps = tt.trace_tag_prefixes(spt.make_smi(drugs_by_name2['Warfarin']))
tt.export_tag_prefixes_pptx(
    steps,
    "mols_text/Warfarin_tag.pptx",
    palette=plp.PALETTE,
    single_atom_palette=PALETTE[-3:]
    # single_atom_color=plp.SINGLE_ATOM_COLOR,
)

In [ ]:
plp.PALETTE = PALETTE
plp.draw_partition(
    drugs_by_name2['Carvedilol'],
    default_atom_label_offset=.3,
    include_default_draw_coords=True,
    image_size=1500,
    atom_label_adjustments={i:{'scale': 0.8} for i in range(45)} | {
        # 1: {"offset": (.35 , -1), 'scale': 0.8},
        # 2: {"offset": (.05, .5), 'scale': 0.8},
        # 8: {"offset": (.2, .05), 'scale': 0.8},
        # 9: {"offset": (.1, -.3), 'scale': 0.8},
    },
    draw_coords={
        (0, 1):{
            'color':None,
            'label':{
                'text':'r<sub>1</sub>',
                'color':'red',
                # 'offset':(-.04, 0)
            }
        },
        (1, 2):{
            'color':None,
            'label':{
                'text':'r<sub>2</sub>',
                'color':'blue',
                'offset':(-.05, 0)
            }
        },
        (0, 1, 2):{
            'color':'green',
            'radius':.8,
            'label':{
                'text':'a<sub>1</sub>',
                'offset':(1.3, -.1)
            }
        }
    }
    # draw_coords_options={
    #     'a1':{'radius':.8}
    # },
    # draw_label_adjustments={
    #     # "r1": {"offset": (.4, -2)},
    #     "r2": {"offset": (0.0, .6)},
    #     "a1": {"offset": (.3, .3)},
    # },
)#.to_widget().write('Carvedilol_colored.svg')

In [ ]:
make_smi(drugs_by_name2['Carvedilol'])

## GEOM

In [ ]:
import plot_bond_orders as pbo
# importlib.reload(pbo)

In [ ]:
samp = dev.read_json('sample_smis.json')

In [ ]:
smi = samp[0]
mol = Molecule.from_string(smi)

In [ ]:
def white_rdkit_lines(drawer):
    options = drawer.drawOptions()
    options.setAtomPalette({-1: (1.0, 1.0, 1.0)})
    options.setSymbolColour((1.0, 1.0, 1.0))
    return drawer

pbo.plot_bonds_by_order(mol, include_hydrogens=True,
                        single_color=plt.prep_color(palette='default', index=0),
                        double_color=plt.prep_color(palette='default', index=1),
                        aromatic_color=plt.prep_color(palette='default', index=3),
                        triple_color=plt.prep_color(palette='default', index=2),                        # atom_style={'stroke':'white'},
                        bond_radius=.51,
                        atom_radii=.42,
                        # atom_style=sty,
                        predraw=white_rdkit_lines,
                        atom_highlights_are_circles=True,
                        color='white',
                        fixed_font_size=30,
                        view_settings={'up_vector':[.75, 1, 0]}
                        ).to_widget().write('figs/bond_coloring_example_compact.svg')

In [ ]:
/Users/Mark/Documents/Postdoc/Projects/Encoding/plot_bond_orders.py

##### Tests

### Statistics

In [ ]:
plot_chain_decomp(
    Molecule.from_string(
        Molecule.from_string(smis[4]).to_string('smi', remove_hydrogens=True)
    ),
    image_size=800,
    # display_atom_numbers=True,
    label_style={'font_size': 12},
    atom_highlights_are_circles=True,
    draw_implicit_hydrogens=False,
    blend_mixed_bonds=False
    # continuous_highlight=True

)

In [ ]:
plot_chain_decomp(
    Molecule.from_string(
        Molecule.from_string(smis[1]).to_string('smi', remove_hydrogens=True)
    ),
    image_size=800,
    # display_atom_numbers=True,
    label_style={'font_size': 12},
    atom_highlights_are_circles=True,
    draw_implicit_hydrogens=False,
    blend_mixed_bonds=False
    # continuous_highlight=True

)

#### Reconstructions

In [ ]:
# validators = importlib.reload(validators)

In [ ]:
mol_data = []
for m in smis:
    if '.' in m: continue
    mol_data.append(
        validators.validate_rmds(m, encoding_lengths=[16, 64], verbose=False, 
                                             confgen_options=dict(random_seed=123232))
    )
np.max([m[16]['compressed']['rmsd'] for m in mol_data])

In [ ]:
np.average([m[16]['compressed']['rmsd'] for m in mol_data])

In [ ]:
np.average([m[16]['precision']['rmsd'] for m in mol_data])

In [ ]:
np.average([m[16]['compressed']['rmsd'] for m in mol_data])

In [ ]:
max_error = np.argmax([m[16]['compressed']['rmsd'] for m in mol_data])

In [ ]:
f1 = mol_data[max_error][16]['compressed']['mols'][0].plot(backend='x3d',
                                        image_size=1000)
mol_data[max_error][16]['compressed']['mols'][1].plot(backend='x3d', figure=f1, highlight_atoms=True, 
                                       include_save_buttons=True)

In [ ]:
f1 = mol_data[max_error][16]['precision']['mols'][0].plot(backend='x3d',
                                        image_size=1000)
mol_data[max_error][16]['precision']['mols'][1].plot(backend='x3d', figure=f1, highlight_atoms=True, 
                                       include_save_buttons=True)

#### Efficiency Comp

In [ ]:
comp_eff_data = {
    (k1, k2): validators.encoding_efficiency_data(mol_data, k1, k2)
    for k1 in (16, 64)
    for k2 in ('compressed', 'precision')
}

In [ ]:
import pprint
pprint.pprint(
{
    k0:{
        k: np.array([np.average(v), np.max(v)])
        for k, v in cd.items()
    }
    for k0, cd in comp_eff_data.items()
}
)

In [ ]:
validators = importlib.reload(validators)

In [ ]:
oeff = validators.other_format_efficiency_data(mol_data, 'xyz', num_prec=3)

In [ ]:
np.average([
    o['rmsd'] for o in oeff
])
    

In [ ]:
np.average([
    o['efficiency'] for o in oeff
])

In [ ]:
np.max([
    o['efficiency'] for o in oeff
])

In [ ]:
3.0 / 4.38888833

##### XYZ Comp

In [ ]:
def check_base16_cartesian_accuracy(mols):
    all_stretch, all_bend, all_dihed, all_diff, all_rmsds = [], [], [], [], []
    if not isinstance(mols, (list, tuple)): mols = [mols]
    for m in mols:
        stretch, bend, dihedral = m.get_bond_graph_internals(concatenate=False, include_fragments=False)
        base_stretch = nput.internal_coordinate_tensors(m.coords * UnitsData.convert("BohrRadius", "Angstroms"), stretch, order=0)[0]
        base_bend = nput.internal_coordinate_tensors(m.coords, bend, order=0)[0]
        base_dihedral = nput.internal_coordinate_tensors(m.coords, dihedral, order=0)[0]
    
        enc_stretch = nput.internal_coordinate_tensors(
            (m.coords * UnitsData.convert("BohrRadius", "Angstroms")).astype('float16'), stretch, order=0)[0]
        enc_bend = nput.internal_coordinate_tensors(
            m.coords.astype('float16'), bend, order=0)[0]
        enc_dihedral = nput.internal_coordinate_tensors(
            m.coords.astype('float16'), dihedral, order=0)[0]
        dd = base_dihedral - enc_dihedral
        mask = np.abs(dd) > np.pi
        enc_dihedral[mask] += np.sign(dd[mask]) * 2*np.pi

        all_stretch.append(base_stretch - enc_stretch)
        all_bend.append(base_bend - enc_bend)
        all_dihed.append(base_dihedral - enc_dihedral)
        all_diff.append((m.coords - m.coords.astype('float16')).flatten())
        all_rmsds.append(np.sqrt(np.linalg.norm(all_diff[-1]) / len(all_diff[-1])))

    all_diff = np.concatenate(all_diff)
    return (
        [np.mean(np.abs(np.concatenate(all_stretch))), np.max(np.abs(np.concatenate(all_stretch)))],
        [np.rad2deg(np.mean(np.abs(np.concatenate(all_bend)))), np.rad2deg(np.max(np.abs(np.concatenate(all_bend))))],
        [np.rad2deg(np.mean(np.abs(np.concatenate(all_dihed)))), np.rad2deg(np.max(np.abs(np.concatenate(all_dihed))))],
        [np.mean(all_rmsds), np.max(all_rmsds)]
    )

In [ ]:
np.mean([
    m[16]['xyz_chars'] / len(m[16]['compressed']['smi'])
    for m in mol_data
])

In [ ]:
np.mean([
    m[64]['xyz_chars'] / len(m[64]['compressed']['smi'])
    for m in mol_data
])

In [ ]:
np.mean([
    len(m[64]['precision']['tag']) / len(m[64]['compressed']['smi'])
    for m in mol_data
])

In [ ]:
np.mean([
    len(m[64]['compressed']['tag']) / len(m[64]['compressed']['smi'])
    for m in mol_data
])

In [ ]:
len(load_data(mol_data, 0, 'mols', 0).remove_hydrogens().atoms) * 6

In [ ]:
import base64

len(
    base64.b16encode(load_data(mol_data, 0, 'mols', 0).remove_hydrogens().coords.astype('float16'))
) / 2

In [ ]:
np.mean([
    len(m[16]['compressed']['tag']) / len(m[16]['compressed']['smi'])
    for m in mol_data
])

In [ ]:
np.mean([
    len(m[16]['compressed']['tag']) / len(m[16]['compressed']['smi'])
    for m in mol_data
])

In [ ]:
np.mean([
    len(m[16]['compressed']['tag']) / len(m[16]['compressed']['smi'])
    for m in mol_data
])

In [ ]:
decoder(mol.coords.astype('float' + str(encoding)).getbuffer())

In [ ]:
2/(2**16) / 2 / 2

In [ ]:
360/(2**15) / 2 / 2

In [ ]:
360/(2**8) / 2

In [ ]:
180/(2**16)

In [ ]:
2/(2**16)

In [ ]:
len(mol_data)

In [ ]:
check_base16_cartesian_accuracy([load_data(mol_data, i, 'mols', 0) for i in range(len(mol_data))])

In [ ]:
np.mean([
    m[64]['compressed']['rmsd']
    for m in mol_data
])

In [ ]:
np.max([
    m[64]['compressed']['rmsd']
    for m in mol_data
])

In [ ]:
np.max([
    m[64]['precision']['rmsd']
    for m in mol_data
])

In [ ]:
np.mean([
    m[64]['precision']['rmsd']
    for m in mol_data
])

In [ ]:
load_data(mol_data, 1, 'rmsd', encoding='precision')

In [ ]:
check_base16_accuracy(load_data(mol_data, 1, 'mols', 0))

#### GZIP

In [ ]:
conf_tagged_smis = []
for s in csmis:
    if '.' not in s:
        cddd = Molecule.from_string(
            s, 'smi',
            num_confs=10
        )
        conf_tagged_smis.append([
            c.to_string('smi', remove_hydrogens=True, include_tag=True)
            for c in cddd
        ])    

In [ ]:
pure_tag_dataset = [s.split("_", 1)[1] for cs in conf_tagged_smis for s in cs]

In [ ]:
len(pure_tag_dataset)

In [ ]:
len(
    gzip.compress("\n".join(pure_tag_dataset).encode())
)

In [ ]:
1 - (94197/136389)

In [ ]:
len(
    "\n".join(pure_tag_dataset)
)

In [ ]:
2 / (2**15)

In [ ]:
(12 * 16) / (12 + 16)

In [ ]:
k = (
    (400 * UnitsData.convert("Wavenumbers", "Hartrees")) ** 2 
) * (((12 * 16) / (12 + 16) * UnitsData.convert("AtomicMassUnits", "ElectronMass"))) / (np.rad2deg(1) ** 2)
k * (.5**2) * UnitsData.convert("Hartrees", "Kilojoules/Mole")

In [ ]:
np.rad2deg(.005)

#### Figure Molecule

In [ ]:
m = Molecule.from_string(r'O=C(Cc1cccc(C(F)(F)F)c1)O/N=C(\C1COc2ccccc2O1)N', 'smi').remove_hydrogens()

In [ ]:
mm = m.take_submolecule(
    list(range(27))
)
# mm = m
mm.to_string('smi', include_tag=True)

In [ ]:
'xDEmPyeppj4AmZk4QKYgOH+u0jcAqJA4/6grP4Cltjd3oA=='
'xDEmPyeppj4AmZk4QKYgOH+u0jcAqJA4/6grP4Cltjd3oH44I5g='
'xDEmPyeppj4AmZk4QKYgOH+u0jcAqJA4/6grP4Cltjd3oH44I5gOOM6e'
'xDEmPyeppj4AmZk4QKYgOH+u0jcAqJA4/6grP4Cltjd3oH44I5gOOM6erTe/sA=='
'xDEmPyeppj4AmZk4QKYgOH+u0jcAqJA4/6grP4Cltjd3oH44I5gOOM6erTe/sFw5f6j9N/+jnzJUpkI//652QVWZqzosl984JKcCOXCu4jiAq1Y4/6kcOQCsrzj/qOA4gKc='
'xDEmPyeppj4AmZk4QKYgOH+u0jcAqJA4/6grP4Cltjd3oH44I5gOOM6erTe/sFw5f6j9N/+jnzJUpkI//652QVWZqzosl984JKcCOXCu4jiAq1Y4/6kcOQCsrzj/qOA4gKcWO4Cq'

In [ ]:
len('xDEmPyeppj4AmZk4QKYgOH+u0jcAqJA4/6grP4Cltjd3oH44I5gOOM6erTe/sFw5f6j9N/+jnzJUpkI//652QVWZqzosl984JKcCOXCu4jiAq1Y4/6kcOQCsrzj/qOA4gKcWO4Cq')

In [ ]:
len('O=C(Cc1cccc(C(F)(F)F)c1)ON=C(C1COc2ccccc2O1)N')

In [ ]:
136 / 45

### Tokenization

##### Conformer Library

In [ ]:
mol_ds = {
    s:Molecule.from_string(s, 'smi', num_confs=10, return_multiconf=True)
    for s in smis if '.' not in s
}
ds = {s:m.to_state() for s,m in mol_ds.items()}
scaff.write_flat_tree('mols.npz', ds)

In [ ]:
ds = {s:m.to_state() for s,m in mol_ds.items()}

In [ ]:
scaff.write_flat_tree('mols.npz', ds)

In [ ]:
ds2 = scaff.read_flat_tree('mols.npz')

In [ ]:
mol2.modify(coords=mol2.coords[5])

##### Scaffolds

In [ ]:
# ds2 = scaff.read_flat_tree('mols.npz')
# mol_ds2 = {s:Molecule.from_state(m) for s,m in ds2.items()}

In [ ]:
# mol = mol_ds2['COC(=O)c1ccc(C(=O)OC)c(NC(=S)NC(=O)c2ccc(OC)c(c2)[N+]([O-])=O)c1']
# mol.modify(coords=mol.coords[5])

In [ ]:
# mol2.modify(coords=mol2.coords[0])

##### Tokens From Data

In [ ]:
def annotation_dist(a):
    return (
        int.from_bytes(a.annotation[:2], byteorder='little')
            if a.atom.index > 0 else 
        -1
    )
def annotation_angle(a):
    return (
        a.annotation[3]
            if a.atom.index > 2 else
        int.from_bytes(a.annotation[2:4])
            if a.atom.index > 1 else 
        -1
    )
def annotation_dihedral(a):
    return (
        a.annotation[2]
            if a.atom.index > 2 else
        -1 
    )
def annotation_zmat(annotation):
    return [
        (a.atom.symbol.capitalize(), 
         a.atom.previous if a.atom.previous is not None else -1, 
         annotation_dist(a), annotation_angle(a), annotation_dihedral(a),
        a.atom.index,
         a.atom.bond_order
        )
        for a in annotation
    ]

In [ ]:
def get_annotation_list(d):
    return {
        'chars': list(SMILESTokenizer().annotate(
            d[16]['compressed']['smi'],
            d[16]['compressed']['tag'],
            [0, 4, 8]
        )),
        'bytes': list(
            SMILESTokenizer().annotate(
                d[16]['compressed']['smi'],
                d[16]['compressed']['bytes'],
                [0, 2, 4]
            )
        ),
        'ints': list(
            SMILESTokenizer().annotate(
                d[16]['compressed']['smi'],
                np.frombuffer(d[16]['compressed']['bytes'], dtype='uint16'),
                [0, 1, 2]
            )
        )
    }

In [ ]:
def sequence_equals(x1, x2, prev1, prev2, cache={}):
    c1, i1, r1, a1, d1, _, b1 = x1
    c2, i2, r2, a2, d2, _, b2 = x2
    #TODO: integrate chain-based caching?
    return (
        c1 == c2 
        and (b1 is None or b2 is None or b1 == b2)
        # and abs(r1 - r2) < 500
        and (
            (len(prev1) == 0 and len(prev2) == 0)
            or abs(d1 - d2) < 15
            or (abs(d1 - d2) - 127) < 10
            or abs(d1 - d2) > 240
        )
    )

def match_counts(al1, al2, match_thresh=4, dedupe_matches=True):
    all_matches = itut.find_tree_overlaps(
        al1,
        al2,
        [None if a[1] < 0 else a[1] for a in al1],
        [None if a[1] < 0 else a[1] for a in al2],
        equals=sequence_equals,
        # mode='best_forest',
        mode='root_alternatives',
        require_contiguous=False
    )

    reduced_groups = [
        m 
        for m in all_matches
        if len(m.matches) >= match_thresh
    ]

    if dedupe_matches:
        _ = []
        match_set_0 = set()
        match_set_1 = set()
        for m in reduced_groups:
            ms_0 = frozenset({i[0] for i in m.matches})
            ms_1 = frozenset({i[1] for i in m.matches})
            if ms_0 not in match_set_0 and ms_1 not in match_set_1:
                _.append(m)
                match_set_0.add(ms_0)
                match_set_1.add(ms_1)
        reduced_groups = _

    group_counts_0 = np.unique([i[0] for m in reduced_groups for i in m.matches], return_counts=True)
    group_counts_1 = np.unique([i[1] for m in reduced_groups for i in m.matches], return_counts=True)

    # group_counts = [
    #     np.unique([i[0] for j in match_indices for i in all_matches[j].matches])
    # ]

    return group_counts_0, group_counts_1, reduced_groups

def mol_sim_from_tokens(al1, al2):
    group_counts_0, group_counts_1, reduced_groups = match_counts(al1, al2)
    return len(group_counts_0[0]) / len(al1), len(group_counts_1[0]) / len(al2), (group_counts_0[0], group_counts_1[0], reduced_groups)

def mol_annot_sim(annotation_1, annotation_2):
    return mol_sim_from_tokens(
        annotation_zmat(annotation_1),
        annotation_zmat(annotation_2)
    )

def mol_matches(annotation_lists, i, j):
    return mol_annot_sim(
        annotation_lists[i]['bytes'],
        annotation_lists[j]['bytes']
    )

def plot_mol_matches(mol_data, i, j, group_data, match_indices=[0, 1]):
    s1, s2, matches = group_data
    return plot_match_comp2(
        load_data(mol_data, i, 'mols', 0).get_embedded_molecule(sel=s1),
        load_data(mol_data, j, 'mols', 0).get_embedded_molecule(sel=s2),
        [matches[i].matches for i in match_indices],
        image_size=500,
        include_save_buttons=True,
        view_settings={'view_distance':15},
        background='white'
    )
    
    

In [ ]:
match_percentages = [
    mol_matches(annotation_lists, i, j)[:2]
    for i in range(len(annotation_lists))
    for j in range(i+1, len(annotation_lists))
]
    

In [ ]:
match_percentages[:2]

In [ ]:
i, j = 0, 1
s1, s2, matches = mol_matches(annotation_lists, i, j)
print(s1, s2)
plot_mol_matches(mol_data, i, j, matches)

In [ ]:
match_indices = [0, 1]
plot_match_comp2(
    load_data(mol_data, 1, 'mols', 0).get_embedded_molecule(sel=np.unique([i[0] for j in match_indices for i in all_matches[j].matches])),
    load_data(mol_data, 4, 'mols', 0).get_embedded_molecule(sel=np.unique([i[1] for j in match_indices for i in all_matches[j].matches])),
    [all_matches[i].matches for i in match_indices],
    image_size=500,
    include_save_buttons=True,
    view_settings={'view_distance':15},
    background='white'
)

In [ ]:
annotation_lists = [
    get_annotation_list(d)
    for d in mol_data
]

In [ ]:
# plot_decomp_comp2(
#     load_data(mol_data, 1, 'mols', 0),
#     load_data(mol_data, 4, 'mols', 0),
#     image_size=500,
#     include_save_buttons=True
# )

In [ ]:
match_indices = [0, 1]
plot_match_comp2(
    load_data(mol_data, 1, 'mols', 0).get_embedded_molecule(sel=np.unique([i[0] for j in match_indices for i in all_matches[j].matches])),
    load_data(mol_data, 4, 'mols', 0).get_embedded_molecule(sel=np.unique([i[1] for j in match_indices for i in all_matches[j].matches])),
    [all_matches[i].matches for i in match_indices],
    image_size=500,
    include_save_buttons=True,
    view_settings={'view_distance':15},
    background='white'
)

In [ ]:
plot_highlight_mol(
    load_data(mol_data, 3, 'mols', 0),
    all_matches[1].matches,
    1
)

In [ ]:
2 / ((2**15) - 1)

In [ ]:
np.rad2deg(2*np.pi / 256) 

In [ ]:
a = np.random.uniform(2.5, 25, 10000000)
np.max(np.abs(a - a.astype('float16')))

In [ ]:
plot_highlight_mol(
    load_data(mol_data, 1, 'mols', 0),
    overlap_lists,
    0
)

In [ ]:
load_data(mol_data, 3, 'mols', 0).plot(
    backend='x3d',
    highlight_atoms=[ol[1] for ol in overlap_lists[1]],
    highlight_styles={'glow':plt.prep_color('green', lighten=-2)}
)

In [ ]:
load_data(mol_data, 1, 'mols', 0)

In [ ]:
[
    (a.atom.symbol.capitalize(), annotation_dist(a), annotation_angle(a), annotation_dihedral(a))
    for a in annotation_lists[0]['bytes']
]

In [ ]:
dist, ang, dihed = ConformerEncoder._split_encoded_streams(
    np.frombuffer(mol_data[0][16]['compressed']['bytes'], dtype='uint16'),
    16,
    pack_angles=True
)

In [ ]:
ConformerEncoder.angle_decoder(
    ang,
    16,
    pack_angles=True
)

In [ ]:
np.frombuffer(
    bytes.fromhex("".join([
        a.annotation[4:6]
        for a in annotation_lists[0]['chars']
        if a.atom.index > 1
    ])),
    dtype='uint8'
)

In [ ]:
angle_chars = [
            a.annotation[6:8]
            for a in annotation_lists[0]['chars']
            if a.atom.index > 1
        ]
angle_chars

In [ ]:
np.frombuffer(
        bytes.fromhex("".join()),
        dtype='uint8'
    )

In [ ]:
ang[1:]

In [ ]:
ConformerEncoder.angle_decoder(
    ang,
    16,
    pack_angles=True
)

In [ ]:
ConformerEncoder.angle_decoder(
    np.array(
        [a.annotation[3]
         for a in annotation_lists[0]['bytes']
         if len(a.annotation) > 2]
    ),
    16,
    pack_angles=True
)[1]

In [ ]:
np.array(
    [a.annotation[2]
     for a in atom_list_bytes
      if len(a.annotation) > 2][1:]
)

## Coley

### Resconst

In [ ]:
def composed_dihedral(d_ijka, d_ijkb): # gives d_ajkb
    d_ijka = np.asanyarray(d_ijka)
    d_ijkb = np.asanyarray(d_ijkb)
    diffs = d_ijkb - d_ijka
    ad = np.asanyarray(np.abs(diffs))
    # print(diffs, d_ijkb,  d_ijka)
    if ad.ndim == 0:
        if ad > np.pi:
            if diffs > 0:
                diffs -= 2 * np.pi
            else:
                diffs += 2 * np.pi
    else:
        mod_pos = np.abs(diffs) > np.pi
        if np.any(mod_pos):
            gz_pos = diffs > 0
            diffs[mod_pos & gz_pos] -= 2*np.pi
            diffs[mod_pos & (~gz_pos)] += 2*np.pi
    return diffs

In [ ]:
d_0123, d_1234, d_0234, d_0124, d_3124 = nput.internal_coordinate_tensors(
    np.random.rand(5, 3),
    [
        (0, 1, 2, 3),
        (1, 2, 3, 4),
        (0, 2, 3, 4),
        (0, 1, 2, 4),
        (3, 1, 2, 4),
    ],
    order=0
)[0]

In [ ]:
np.rad2deg([d_0123, d_1234, d_0234, d_0124, d_3124])

In [ ]:
np.rad2deg(composed_dihedral(d_0123, d_0124)) + 360

In [ ]:
np.rad2deg(nput.internal_coordinate_tensors(
    np.random.rand(5, 3),
    [
        (0, 1, 2, 3),
        (1, 2, 3, 4),
        (0, 2, 3, 4),
        (0, 1, 2, 4),
    ],
    order=0
)[0])

### Model Tests

In [ ]:
import numpy as np

def get_sub_zmatrix(zmatrix: np.ndarray, atom_set):
    """
    Returns the minimal self-contained sub-Z-matrix connecting atom_set:
    ONE valid (M, 4) Z-matrix array where every reference used by any row
    is ITSELF a row present in the same returned array, all the way back to
    genuine anchor atoms (rows with no_ref_sentinel refs) -- containing
    atom_set plus exactly whatever additional atoms are needed to satisfy
    that self-containment, nothing more.

    zmatrix: (N, 4) int np.ndarray -- column 0 is atom_idx, columns 1-3 are
        (bond_ref, angle_ref, dihedral_ref), padded with `no_ref_sentinel`
        where a reference doesn't exist.

    Returns: (M, 4) sub-array, in original chain order, ready to hand to
    your own placement code as a standalone, complete Z-matrix.
    """
    atom_idx_col = zmatrix[:, 0]
    row_by_atom = {int(atom_idx_col[i]): zmatrix[i, 1:4] for i in range(zmatrix.shape[0])}

    def get_refs(atom_idx):
        refs = row_by_atom.get(atom_idx)
        if refs is None:
            return ()
        return refs

    chain_atoms = set()
    stack = [int(a) for a in atom_set]
    while stack:
        atom = stack.pop()
        if atom in chain_atoms:
            continue
        chain_atoms.add(atom)
        for ref in get_refs(atom):
            if ref not in chain_atoms and ref > 0:
                stack.append(ref)

    mask = np.array([int(a) in chain_atoms for a in atom_idx_col])
    return atom_idx_col[mask,], zmatrix[mask]

In [ ]:
mol = Molecule.from_string(smis[0])

In [ ]:
zm = np.array(mol.get_canonical_zmatrix())

In [ ]:
get_sub_zmatrix(zm, [5, 15])

In [ ]:
mol.plot(highlight_atoms=_30[0], backend='x3d')

In [ ]:
# from McUtils.ExternalPrograms import SMILESSupplier
# import pandas as pd
# import numpy as np

# ds = pd.read_pickle('/Users/Mark/Documents/Postdoc/Projects/Encoding/test_RS_classification_enantiomers_MOL_69719_11680_5840.pkl')

# mols = [ 
#     Molecule.from_rdmol(mmm.GetConformer()).to_string('smi', remove_hydrogens=True, include_tag=True) 
#     for mmm in ds['rdkit_mol_cistrans_stereo']
# ]

# with open('/Users/Mark/Desktop/coley_classification_test_tags.smi', 'w+') as bbbbb:
#     bbbbb.write("\n".join(mols))

# with SMILESSupplier('/Users/Mark/Desktop/coley_classification_test_tags.smi',
#                    metadata_arrays={'ld':ds['RS_label_binary']}
#                    ) as (supp, _):
#     supp.write_database_index('coley_classification_test_data.tar')
    

In [ ]:
supp.write_database_index

In [ ]:
# ds = pd.read_pickle('/Users/Mark/Documents/Postdoc/Projects/Encoding/test_RS_classification_enantiomers_MOL_69719_11680_5840.pkl')

# mols = [ 
#     Molecule.from_rdmol(mmm.GetConformer()).to_string('smi', remove_hydrogens=True, include_tag=True) 
#     for mmm in ds['rdkit_mol_cistrans_stereo']
# ]

# with open('/Users/Mark/Desktop/coley_classification_test_tags.smi', 'w+') as bbbbb:
#     bbbbb.write("\n".join(mols))

In [ ]:
from McUtils

In [ ]:
ds['RS_label_binary']

In [ ]:
import numpy as np

a = np.load('/Users/Mark/Documents/Postdoc/Projects/Encoding/coley_contrastive_test_data.npz')

with open('coley_contrastive_test_data.smi', 'w+') as f:
    f.write("\n".join(a['tags']))

In [ ]:
import numpy as np

a = np.load('/Users/Mark/Documents/Postdoc/Projects/Encoding/coley_contrastive_test_validation.npz')

with open('coley_contrastive_validation_data.smi', 'w+') as f:
    f.write("\n".join(a['tags']))

In [ ]:
from McUtils.ExternalPrograms import SMILESSupplier

In [ ]:
import numpy as np

a = np.load('/Users/Mark/Documents/Postdoc/Projects/Encoding/coley_contrastive_test_data.npz')

with open('coley_contrastive_test_data.smi', 'w+') as f:
    f.write("\n".join(a['tags']))

smi_file = 'coley_contrastive_test_data.smi'
db_file = 'coley_contrastive_test_data.tar'

# vendor = SMILESSupplier(smi_file)
# with vendor:
#     vendor.write_database_index(db_file, overwrite=True)

In [ ]:
from Psience.Molecools import Molecule

uuh = SMILESSupplier.from_line_index_database(db_file)
m = Molecule.from_string(uuh.find_smi(99548), 'smi')

In [ ]:
m

In [ ]:
Molecule.from_string(
    'Fc1c(/C(=C2/C(=C(\c3c(F)c(F)c(C#N)c(F)c3F)C#N)/C2=C\c2c(F)c(F)c(C#N)c(F)c2F)C#N)c(F)c(F)c(C#N)c1F_1DVFObCrjD//qiQ0Kqy6PILQsDUyq8A+L6tGOT+q0TX/q0Y5f6rUNX+pRjkAqnc7f6pFKtv/RTn/qtQ1gKtFOb6q0jUAq4I7rqlIKiz/8jUB1iE0/9YdPX+zRTnVqtM1/6tJOX+q1DV/qUY5Nap2O3+qSCpW/0Q5/6rVNYCrRjlWqtM1AKtDPKqnRyqB/kU5f6rTNYCrRjn/qtM1f6lGOf+qeDuAqkUqZP9IOX+p1jUAqQ==',
    'smi'
)

In [ ]:
from McUtils.ExternalPrograms import SMILESSupplier

# import numpy as np

# a = np.load('/Users/Mark/Documents/Postdoc/Projects/Encoding/coley_contrastive_data_tags.npz')

# with open('coley_contrastive_data_tags.smi', 'w+') as f:
#     f.write("\n".join(a['tags']))

smi_file = 'coley_contrastive_data_tags.smi'
db_file = 'coley_contrastive_data_tags.tar'

vendor = SMILESSupplier(smi_file)
with vendor:
    vendor.write_database_index(db_file, overwrite=True)

In [ ]:
from McUtils.ExternalPrograms import SMILESSupplier

with SMILESSupplier.from_line_index_database(
    '/Users/Mark/Documents/Postdoc/Projects/Encoding/coley_contrastive_data_tags.tar'
) as (supp, _):
    print(len(supp))

In [ ]:
import numpy as np

a = np.zeros(1000)

In [ ]:
isinstance(a, np.memmap)

In [ ]:
with open('/Users/Mark/Documents/Postdoc/Projects/Encoding/coley_contrastive_data_tags/lines.smi') as wtf:
    n = 0
    for _ in wtf:
        n+=1
    print(n)

In [ ]:
len(uuh)

In [ ]:
from McUtils.ExternalPrograms import SMILESSupplier

# import numpy as np

# a = np.load('/Users/Mark/Documents/Postdoc/Projects/Encoding/coley_contrastive_data_tags.npz')

# with open('coley_contrastive_data_tags.smi', 'w+') as f:
#     f.write("\n".join(a['tags']))

smi_file = 'coley_contrastive_validation_data.smi'
db_file = 'coley_contrastive_validation_data.tar'

vendor = SMILESSupplier(smi_file)
with vendor:
    vendor.write_database_index(db_file, overwrite=True)

In [ ]:
from McUtils.ExternalPrograms import SMILESSupplier
import numpy as np

# a = np.load('/Users/Mark/Documents/Postdoc/Projects/Encoding/coley_contrastive_data_tags.npz')

# with open('coley_contrastive_data_tags.smi', 'w+') as f:
#     f.write("\n".join(a['tags']))

smi_file = 'coley_RS_classification_data.smi'
db_file = 'coley_RS_classification_data.tar'

vendor = SMILESSupplier(smi_file, metadata_arrays={'rs':np.load('coley_RS_classification_data_ld.npy')})
with vendor:
    vendor.write_database_index(db_file, overwrite=True)

In [ ]:
from McUtils.ExternalPrograms import SMILESSupplier
from Psience.Molecools import Molecule

db_file = '/Users/Mark/Documents/Postdoc/Projects/Encoding/optimal-conformer-encodings/tests/TestData/coley_RS_classification_validation_data.tar'
with SMILESSupplier.from_line_index_database(db_file) as (supp, _):
    print(len(supp))
    smi, optical_rotation_data = supp.find_smi(1543)
    mol = Molecule.from_string(smi, 'smi')
print(optical_rotation_data)
mol.plot(backend='x3d')

#### Training Curves

In [ ]:
import sys, os
sys.path.insert(0, 'optimal-conformer-encodings')

import importlib

import numpy as np
from McUtils.ExternalPrograms import SMILESSupplier, RDMolecule, SMILESTokenizer, ConformerEncoder
from McUtils.Data import UnitsData
from Psience.Molecools import Molecule
import McUtils.Iterators as itut
import McUtils.Scaffolding as scaff
import McUtils.Plots as plt
import McUtils.Coordinerds as coordops
import McUtils.Numputils as nput
import McUtils.Jupyter as interactive

import itertools

import ocen.validators as validators

import ocen.models.analysis as analysis

In [ ]:
split = 'train'
key = 'acc'
per_epoch = True
f0 = analysis.plot_training_curve('train-19618966.out',
                                  split=split, key=key, per_epoch=per_epoch,
                                  common_settings=dict(color=plt.prep_color(palette='default', index=0)),
                                  label='Complete',
                                  plot_legend=True,
                                  legend_style={
                                      'loc':'upper left',
                                      'frameon':False,
                                      'ncol':3,
                                      'handletextpad':.3,
                                      'columnspacing':1,
                                      'bbox_to_anchor':[0, 1.08],
                                      'fontsize':12
                                  },
                                  
                                  # plot_range=[[0, 20_000], None],
                                  axes_labels=['Batch', 'Loss' if key == 'loss' else 'Accuracy'],
                                  image_size=[440, 330],
                                  # ticks=[[0, 10_000, 20_000, 30_000], None]
                                  )
# analysis.plot_training_curve('train-19586043.out', split=split, key=key,
#                              common_settings=dict(color=plt.prep_color(palette='default', index=0), linestyle='dotted'),
#                              label=None,
#                              figure=f0)
analysis.plot_training_curve('train-19632785.out', split=split, key=key, per_epoch=per_epoch,
                             common_settings=dict(color=plt.prep_color(palette='default', index=0)),
                             label=None,
                             figure=f0)
analysis.plot_training_curve('train-19657116.out', split=split, key=key, per_epoch=per_epoch,
                             common_settings=dict(color=plt.prep_color(palette='default', index=0)),
                             label=None,
                             figure=f0)
analysis.plot_training_curve('train-19722649.out', split=split, key=key, per_epoch=per_epoch,
                             common_settings=dict(color=plt.prep_color(palette='default', index=0)),
                             label=None,
                             figure=f0)

analysis.plot_training_curve('train-19622477.out', split=split, key=key, per_epoch=per_epoch,
                             common_settings=dict(color=plt.prep_color(palette='default', index=2)),
                             label='Minimal',
                             figure=f0)
# analysis.plot_training_curve('train-19586042.out', split=split, key=key,
#                              common_settings=dict(color=plt.prep_color(palette='default', index=2), linestyle='dotted'),
#                              label=None,
#                              figure=f0)
analysis.plot_training_curve('train-19632784.out', split=split, key=key, per_epoch=per_epoch,
                             common_settings=dict(color=plt.prep_color(palette='default', index=2)),
                             label=None,
                             figure=f0)
analysis.plot_training_curve('train-19657095.out', split=split, key=key, per_epoch=per_epoch,
                             common_settings=dict(color=plt.prep_color(palette='default', index=2)),
                             label=None,
                             figure=f0)
analysis.plot_training_curve('train-19723542.out', split=split, key=key, per_epoch=per_epoch,
                             common_settings=dict(color=plt.prep_color(palette='default', index=2)),
                             label=None,
                             figure=f0)

analysis.plot_training_curve('train-19625549.out', split=split, key=key, per_epoch=per_epoch,
                             common_settings=dict(color=plt.prep_color(palette='default', index=1)),
                             label='Augmented',
                             figure=f0)
analysis.plot_training_curve('train-19590925.out', split=split, key=key, per_epoch=per_epoch,
                             common_settings=dict(color=plt.prep_color(palette='default', index=1)),
                             label=None,
                             figure=f0)
analysis.plot_training_curve('train-19632787.out', split=split, key=key, per_epoch=per_epoch,
                             common_settings=dict(color=plt.prep_color(palette='default', index=1)),
                             label=None,
                             figure=f0)
analysis.plot_training_curve('train-19657115.out', split=split, key=key, per_epoch=per_epoch,
                             common_settings=dict(color=plt.prep_color(palette='default', index=1)),
                             label=None,
                             figure=f0)
analysis.plot_training_curve('train-19723543.out', split=split, key=key, per_epoch=per_epoch,
                             common_settings=dict(color=plt.prep_color(palette='default', index=1)),
                             label=None,
                             figure=f0)

analysis.plot_training_curve('train-19734187.out', split=split, key=key, per_epoch=per_epoch,
                             common_settings=dict(color=plt.prep_color(palette='default', index=1)),
                             label=None,
                             figure=f0)
analysis.plot_training_curve('train-19734203.out', split=split, key=key, per_epoch=per_epoch,
                             common_settings=dict(color=plt.prep_color(palette='default', index=1)),
                             label=None,
                             figure=f0)
analysis.plot_training_curve('train-19737133.out', split=split, key=key, per_epoch=per_epoch,
                             common_settings=dict(color=plt.prep_color(palette='default', index=4)),
                             label=None,
                             figure=f0)
analysis.plot_training_curve('train-19737134.out', split=split, key=key, per_epoch=per_epoch,
                             common_settings=dict(color=plt.prep_color(palette='default', index=3)),
                             label=None,
                             figure=f0)

f0.savefig(f'{split}_progress' + ("_loss" if key == "loss" else "") + ".svg")
f0

In [ ]:
analysis = importlib.reload(analysis)

In [ ]:
analysis.plot_training_groups(
    {
        'Complete': [
            'train-19618966.out',
            'train-19657116.out',
            'train-19632785.out',
            'train-19722649.out'
        ],
        'Minimal': [
            'train-19622477.out',
            'train-19632784.out',
            'train-19657095.out',
            'train-19723542.out'
        ],
        'Augmented': [
            'train-19625549.out',
            'train-19632787.out',
            'train-19657115.out',
            'train-19723543.out',
            'train-19734187.out',
            'train-19734203.out',
            ('train-19737133.out', 
             dict(color=plt.prep_color(palette='default', index=3), linestyle=None)),
            ('train-19737134.out', 
             dict(color=plt.prep_color(palette='default', index=3), linestyle='dashed')),
        ],
    },
    split='train',
    key='acc',
    x_source='Batch',
    per_epoch=True,
    image_size=[440, 300],
    padding=[[45, 0], [35, 15]],
    legend_style={
                        'loc': 'upper center',
                        'frameon': False,
                        'ncol': 3,
                        'handletextpad': .3,
                        # 'columnspacing': 2,
                        'bbox_to_anchor': [.5, 1.08],
                        'fontsize': 12
                    }
)#.savefig('figs/training_curves.svg')

In [ ]:
analysis.plot_training_groups(
    {
        'Complete': [
            'train-19657116.out',
            'train-19632785.out',
            'train-19722649.out'
        ],
        'Minimal': [
            'train-19632784.out',
            'train-19657095.out',
            'train-19723542.out'
        ],
        'Augmented': [
            'train-19632787.out',
            'train-19737133.out',
            'train-19657115.out',
            'train-19723543.out',
            'train-19734187.out',
            'train-19734203.out',
            'train-19737134.out'
        ]
    },
    split='train',
    key='rate',
    x_source='time',
    per_epoch=True
)

In [ ]:
split = 'train'
key = 'rate'
x_source = 'time'
per_epoch = True
f0 = analysis.plot_training_curve('train-19632785.out',
                                  split=split, key=key, x_source=x_source, per_epoch=per_epoch,
                                  common_settings=dict(color=plt.prep_color(palette='default', index=0)),
                                  label='Complete',
                                  plot_legend=True,
                                  legend_style={
                                      'loc':'upper left',
                                      'frameon':False,
                                      'ncol':3,
                                      'handletextpad':.3,
                                      'columnspacing':1,
                                      'bbox_to_anchor':[0, 1.08],
                                      'fontsize':12
                                  },
                                  
                                  # plot_range=[[0, 20_000], None],
                                  axes_labels=['Time (s)', 
                                               (
                                                   'Loss' if key == 'loss' else 'Accuracy'],
                                  image_size=[440, 330],
                                  # ticks=[[0, 10_000, 20_000, 30_000], None]
                                  )

analysis.plot_training_curve('train-19657095.out', split=split, key=key, x_source=x_source, per_epoch=per_epoch,
                             common_settings=dict(color=plt.prep_color(palette='default', index=2)),
                             label='Minimal',
                             figure=f0)
f0

In [ ]:
analysis = importlib.reload(analysis)

In [ ]:
analysis.extract_losses(
    analysis.parse_log('train-19737134.out'),
    per_epoch=True
)

In [ ]:
from Psience.Molecools import Molecule

Molecule.

In [ ]:
huh = analysis.extract_losses(analysis.parse_log('train-19586042.out'))
groups = itut.group_by(huh, key=lambda x: (x['epoch'], x['split']))
curves = {k: (
    [h['x'] for h in v],
    [h['loss'] for h in v]
) for k, v in groups.items()
}
f0 = plt.plot_multi(
    *[
        {
            'x': v[0],
            'y': v[1],
            'label': k
        }
        for k, v in curves.items()
        if 'train' in k
    ]
)

huh = analysis.extract_losses(analysis.parse_log('train-19586043.out'))
groups = itut.group_by(huh, key=lambda x: (x['epoch'], x['split']))
curves = {k: (
    [h['x'] for h in v],
    [h['loss'] for h in v]
) for k, v in groups.items()
}
plt.plot_multi(
    *[
        {
            'x': v[0],
            'y': v[1],
            'label': k
        }
        for k, v in curves.items()
        if 'train' in k
    ],
    figure=f0
)

In [ ]:
huh = extract_losses(parse_log('train-19590925.out'))
groups = itut.group_by(huh, key=lambda x: (x['epoch'], x['split']))
curves = {k: (
    [h['x'] for h in v],
    [h['loss'] for h in v]
) for k, v in groups.items()
}
plt.plot_multi(
    *[
        {
            'x': v[0],
            'y': v[1],
            'label': k
        }
        for k, v in curves.items()
        if 'train' in k
    ]
)